# Notebook-backed package loader
Purpose: install an explicit import hook for **definition-only** `ahr-export` cells.
Inputs: project root containing `fmm_jepa/*.ipynb`; outputs: importable `fmm_jepa` and `fmm_tests` namespaces.
Runtime: less than one second. No experiment or training runs on import. No checkpoints.
Resume: rerun this definition cell, then `install_notebook_package(project_root)` in a fresh kernel.
This is the single loader implementation; consumers execute this tagged cell and call the installer.

In [ ]:
from pathlib import Path
import ast
import importlib.abc
import importlib.util
import json
import sys
import types

PACKAGE_VERSION = "0.1.0"
CONFIG_SCHEMA = "fmm-jepa-config/v1"
ARTIFACT_SCHEMA = "fmm-jepa-artifact/v1"

class TaggedNotebookLoader(importlib.abc.Loader):
    def __init__(self, path):
        self.path = Path(path)
    def create_module(self, spec):
        return None
    def exec_module(self, module):
        document = json.loads(self.path.read_text())
        cells = [c for c in document["cells"] if c["cell_type"] == "code" and "ahr-export" in c.get("metadata", {}).get("tags", [])]
        if not cells:
            raise ImportError(f"No ahr-export definitions in {self.path}")
        module.__file__ = str(self.path)
        for index, cell in enumerate(cells):
            source = "".join(cell["source"])
            exec(compile(source, f"{self.path}#ahr-export-{index}", "exec"), module.__dict__)

class TaggedNotebookFinder(importlib.abc.MetaPathFinder):
    def __init__(self, root):
        self.root = Path(root).resolve()
    def find_spec(self, fullname, path=None, target=None):
        pieces = fullname.split(".")
        if len(pieces) != 2 or pieces[0] not in ("fmm_jepa", "fmm_tests"):
            return None
        folder = "fmm_jepa" if pieces[0] == "fmm_jepa" else "tests"
        filename = self.root / folder / f"{pieces[1]}.ipynb"
        if filename.is_file():
            return importlib.util.spec_from_loader(fullname, TaggedNotebookLoader(filename), origin=str(filename))
        return None

def install_notebook_package(project_root):
    root = Path(project_root).expanduser().resolve()
    if not (root / "fmm_jepa" / "bootstrap.ipynb").is_file():
        raise FileNotFoundError(f"Not an FMM-JEPA project root: {root}")
    for existing in sys.meta_path:
        if getattr(existing, "root", None) == root and type(existing).__name__ == "TaggedNotebookFinder":
            return root
    for name, directory in (("fmm_jepa", "fmm_jepa"), ("fmm_tests", "tests")):
        if name in sys.modules:
            previous = getattr(sys.modules[name], "__project_root__", None)
            if previous != str(root):
                raise ImportError(f"Namespace {name} already belongs to {previous}; restart kernel")
        else:
            package = types.ModuleType(name)
            package.__path__ = [str(root / directory)]
            package.__project_root__ = str(root)
            package.__version__ = PACKAGE_VERSION
            sys.modules[name] = package
    sys.meta_path.insert(0, TaggedNotebookFinder(root))
    return root
